# Session 10 · Exercises

*Course material, licence CC BY 4.0.*

Work in pairs. The lecturer chooses which exercises to set; there are more than fit into 30 minutes. *Paper* and *Predict* tasks are answered in the markdown cell below the task, before any code is run; *Code* tasks in the empty code cells. Exercises marked *Stretch* are for pairs who finish early.

## Setup

Run these cells once before the exercises that use data.

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

# Short-stay listings with a price, as in the price-model case study of this session
listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
bnb = listings[(listings["minimum_nights"] < 28) & listings["price"].between(10, 1000)].reset_index(drop=True)
bnb["dist_km"] = np.hypot((bnb["latitude"] - 52.5219) * 111.2,      # km to Alexanderplatz
                          (bnb["longitude"] - 13.4132) * 68.0)

num = ["accommodates", "bedrooms", "beds", "bathrooms", "dist_km", "minimum_nights",
       "availability_365", "number_of_reviews", "review_scores_rating"]
cat = ["room_type", "district", "property_type"]
y, price, hosts = np.log(bnb["price"]), bnb["price"], bnb["host_id"]
folds = list(GroupKFold(5).split(bnb, groups=hosts))      # the same host-grouped folds for every model
print(len(bnb), "listings,", hosts.nunique(), "hosts")

## Part 1 · From one tree to an ensemble

### 1.1 · Gini by hand

*Paper, 10 min*

> A node with 6 churners and 4 stayers. What is its Gini impurity?
>
> Ten listings, and whether each was busy last year (1 = busy):
>
> | Listing | Guests | Room | Busy |
> |---|---|---|---|
> | 1 | 1 | private | 0 |
> | 2 | 2 | private | 0 |
> | 3 | 2 | private | 1 |
> | 4 | 2 | entire | 1 |
> | 5 | 3 | entire | 1 |
> | 6 | 4 | entire | 1 |
> | 7 | 4 | private | 0 |
> | 8 | 5 | entire | 1 |
> | 9 | 6 | entire | 0 |
> | 10 | 8 | entire | 0 |
>
> Find the best first split: compute the weighted Gini impurity of the split on *Room* and of every threshold on *Guests*. Which split does a tree choose?

*Source:* [scikit-learn: Decision trees, mathematical formulation](https://scikit-learn.org/stable/modules/tree.html); the table of ten listings is an original exercise for this course.

Your answer:

### 1.2 · Twenty questions

*Class, 10 min*

> One student thinks of an object; the class asks yes/no questions. Why is "Is it alive?" a good first question?

*Source:* [Twenty questions, Wikipedia](https://en.wikipedia.org/wiki/Twenty_questions)

Your answer:

### 1.3 · XOR

*Paper, 10 min*

> Four points of XOR: (0, 0) and (1, 1) belong to class 0, (0, 1) and (1, 0) to class 1. Which first split improves the impurity? Can a tree of depth 2 classify all four points?

*Source:* [Decision tree learning, Wikipedia](https://en.wikipedia.org/wiki/Decision_tree_learning)

Your answer:

### 1.4 · Can a tree extrapolate?

*Predict, then Code, 10 min*

> A regression tree is trained on Berlin listings for 1–6 guests. What does it predict for 16 guests? Write down your prediction and draw the predicted price against the number of guests.
>
> Then check it: fit a `DecisionTreeRegressor(max_depth=4)` on `bnb` rows with `accommodates <= 6`, using `accommodates` as the only feature and `price` as the target, and predict for 1 to 16 guests. Compare with a linear regression on the same rows.

*Source:* original exercise for this course; background: [scikit-learn example: Decision tree regression](https://scikit-learn.org/stable/auto_examples/tree/plot_tree_regression.html)

Your answer:

### 1.5 · Wisdom of the crowd

*Class, 10 min*

> The lecturer shows one Berlin listing on screen. Everyone writes down its nightly price on a slip of paper. Compare the class average with the individual guesses: how many of you were closer to the true price than the average?

*Source:* [Galton (1907), "Vox populi", *Nature* 75](https://doi.org/10.1038/075450a0)

Your answer:

### 1.6 · Majority vote

*Paper, 10 min*

> Five independent models, each 60 % accurate. How accurate is their majority vote? And if all five models are identical?

*Source:* [Condorcet's jury theorem, Wikipedia](https://en.wikipedia.org/wiki/Condorcet%27s_jury_theorem)

Your answer:

### 1.7 · Boosting by hand

*Paper, 15 min*

> Five listings:
>
> | Guests | 1 | 2 | 3 | 4 | 6 |
> |---|---|---|---|---|---|
> | Price (€) | 40 | 50 | 70 | 90 | 150 |
>
> Round 1 predicts the mean price. Compute the residuals. Round 2 fits the residuals with one split on *Guests* (try every threshold; choose the one with the smallest sum of squared residuals). Add the two predictions (learning rate 1) and compute the new residuals. What would change with a learning rate of 0.5?

*Source:* [Friedman (2001), "Greedy function approximation: a gradient boosting machine", *Annals of Statistics*](https://doi.org/10.1214/aos/1013203451)

Your answer:

### 1.8 · Free validation

*Discussion, 5 min*

> Why does a random forest get an out-of-bag score for free? Connect it to the 36.8 % of Session 7.

*Source:* [Breiman (2001), "Random forests", *Machine Learning* 45](https://doi.org/10.1023/A:1010933404324)

Your answer:

### 1.9 · A price tree and a forest

*Code, pairs, 20 min*

> Predict the log price `y` of the Berlin listings from the numeric columns `num` (setup cell).
>
> 1. Fit `DecisionTreeRegressor(max_depth=d, min_samples_leaf=5)` for `d` from 1 to 15 and compute the training and the cross-validated R² on the host-grouped `folds` (`cross_validate(..., cv=folds, return_train_score=True)`). Plot both against the depth. Which depth do you choose?
> 2. Score a `RandomForestRegressor(n_estimators=200, min_samples_leaf=3, n_jobs=-1)` on the same folds. How much better is it than the best single tree?
> 3. *Stretch.* Fit the chosen tree on 10 bootstrap samples of the listings and print the first two splits each time (`export_text(..., max_depth=1)`). What do you conclude about the stability of single trees?

*Source:* original exercise for this course; background: [scikit-learn: Decision trees](https://scikit-learn.org/stable/modules/tree.html) and [random forests](https://scikit-learn.org/stable/modules/ensemble.html#random-forests)

## Part 2 · Gradient boosting in practice and interpretation

### 2.1 · Class weights

*Predict, 5 min*

> A LightGBM churn model is trained once without weights and once with `scale_pos_weight=3` (each churner counts three times). Predict, before running anything: does ROC AUC go up, down or stay about the same? And the recall and the precision at the default threshold of 0.5?

*Source:* original exercise for this course; background: [LightGBM parameters: `scale_pos_weight`](https://lightgbm.readthedocs.io/en/stable/Parameters.html)

Your answer:

### 2.2 · A SHAP explanation in euros

*Paper, 10 min*

> A LightGBM model predicts the **log** nightly price. For one listing, the SHAP base value is 4.50, and the contributions are +0.30 (accommodates), −0.20 (distance to Alexanderplatz) and +0.10 (room type); all other contributions are zero.
>
> 1. What is the predicted log price, and the predicted price in euros?
> 2. By what factor does the distance change the price of this listing, compared with the base value?
> 3. A host reads the explanation and says: "If my flat were closer to the centre, the price would rise by 22 %." Is this a correct reading of SHAP values? Why or why not?

*Source:* original exercise for this course; background: [Molnar, *Interpretable Machine Learning*, SHAP](https://christophm.github.io/interpretable-ml-book/shap.html)

Your answer:

### 2.3 · Replace the current model?

*Paper, 10 min*

> The current price model and a challenger were scored on the same five host-grouped folds (MAE in € per night):
>
> | Fold | 1 | 2 | 3 | 4 | 5 |
> |---|---|---|---|---|---|
> | Current model | 52.1 | 55.4 | 49.8 | 57.0 | 53.2 |
> | Challenger | 51.0 | 55.9 | 48.1 | 56.2 | 52.5 |
>
> Compute the paired differences, their mean and how many folds the challenger wins. The challenger needs 40 times longer to train and a new library in production. Would you replace the current model? What else would you want to know?

*Source:* original exercise for this course; background: [Bernardi, Mavridis & Estevez (2019), "150 successful machine learning models", KDD](https://doi.org/10.1145/3292500.3330744)

Your answer:

### 2.4 · Native categories or one-hot?

*Code, 15 min*

> Score LightGBM (`LGBMRegressor(n_estimators=300, learning_rate=0.05, verbose=-1)`) on the log price with the columns `num + cat`, on the host-grouped `folds`, in two versions:
>
> 1. `cat` as pandas `category` dtype (native categorical splits);
> 2. `cat` one-hot encoded with `pd.get_dummies`.
>
> Report the cross-validated MAE in euros (`np.exp` of the predictions) and the fit time of both. Is the difference larger than the spread across the folds?

*Source:* original exercise for this course; background: [scikit-learn example: Categorical feature support in gradient boosting](https://scikit-learn.org/stable/auto_examples/ensemble/plot_gradient_boosting_categorical.html)

### 2.5 · Early stopping and the learning rate

*Code, 15 min*

> Split the hosts once into training (80 %) and validation (20 %) with `GroupShuffleSplit`. Train `LGBMRegressor(n_estimators=2000, verbose=-1)` on the training hosts for the learning rates 0.3, 0.1 and 0.03, with `eval_set` on the validation hosts and the callback `lightgbm.early_stopping(50)`.
>
> For each learning rate, report the best number of trees (`best_iteration_`) and the validation L2 loss, and plot the validation curves (`evals_result_`). Why must the validation hosts come from the training data and not from the test data?

*Source:* original exercise for this course; background: [LightGBM: Parameters Tuning](https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html)

### 2.6 · A random column

*Code, pairs, 15 min, Stretch*

> Add a column `random` of standard normal noise to `bnb[num]`. Fit a `RandomForestRegressor(n_estimators=200, n_jobs=-1)` on the training hosts of the previous task.
>
> 1. Rank the features by impurity-based importance (`feature_importances_`). Where does `random` end up?
> 2. Rank them by permutation importance on the validation hosts (`sklearn.inspection.permutation_importance`). Where does `random` end up now?
> 3. Compute the permutation importance on the training hosts as well. Which features look much more important on the training data, and what does that tell you?

*Source:* original exercise for this course; background: [scikit-learn example: Permutation importance vs random forest feature importance](https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance.html)